# §3.2 — DBSCAN Clustering & Filtering

Produces `filtered_clusters.csv` (the 14,582-cluster working dataset).

**Inputs:** `data/clusters_with_distance.csv`  
**Outputs:** `data/filtered_clusters.csv`, Figure 1 (spatial extent map)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from clustering import run_dbscan, filter_clusters
from metrics.shape import compute_all_shape_metrics

In [ ]:
# --- Load tree-level data ---
df = pd.read_csv('../data/clusters_with_distance.csv')
print(f'Loaded {len(df):,} trees')
df.head()

In [ ]:
# --- DBSCAN: eps=20 m, min_samples=3 ---
df = run_dbscan(df, eps=20, min_samples=3)
n_raw = df[df.cluster_id != -1]['cluster_id'].nunique()
print(f'Raw clusters: {n_raw:,}')   # expect ~43,539

In [ ]:
# --- Compute cluster-level shape metrics (needed for filtering) ---
print('Computing shape metrics (takes a few minutes)...')
cluster_df = compute_all_shape_metrics(df[df.cluster_id != -1], cluster_col='cluster_id')
print(f'Metrics computed for {len(cluster_df):,} clusters')

In [ ]:
# --- Filter: >=5 trees, area >=100 m², elongation (aspect_ratio) <=10 ---
filtered = filter_clusters(cluster_df.reset_index(),
                           min_trees=5, min_area=100.0, max_elongation=10.0)
print(f'Filtered clusters: {len(filtered):,}')   # expect ~14,582
filtered.to_csv('../data/filtered_clusters.csv', index=False)
print('Saved → data/filtered_clusters.csv')

In [ ]:
# --- Figure 1: spatial extent map ---
fig, ax = plt.subplots(figsize=(10, 8), dpi=150)
ax.scatter(filtered['cx'], filtered['cy'], s=2, alpha=0.3, color='red', label='Cluster centroids')
ax.set_xlabel('Easting (m)', fontsize=12)
ax.set_ylabel('Northing (m)', fontsize=12)
ax.set_title('Spatial extent of dead-tree clusters (Finland, EPSG:3067)', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../figures/fig1_spatial_extent.png', dpi=300)
plt.show()